# LegalGPT: train the 1.5B on exactly the 3B/7B training data (Colab L4, ~20 min)

So the three model sizes are compared fairly, the 1.5B learns from **the same examples and split**
as the 3B: `MyDrive/legalgpt/lora/v0.7-3b-l4/examples.jsonl`, written by the earlier run.

1. Upload the **new** `legalgpt_colab.zip` to `MyDrive/legalgpt/`, replacing the old one.
2. **Runtime → Change runtime type → L4 GPU**, then **Run all** (about 20-30 min, roughly 2-3 units).
3. Download the folder `MyDrive/legalgpt/lora/v0.7-1.5b-l4` (without `_trainer`) to the laptop's
   `data/models/lora/`. The benchmark runs on the laptop.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, shutil, zipfile
DRIVE, REPO = "/content/drive/MyDrive/legalgpt", "/content/legaltech"
EXAMPLES = f"{DRIVE}/lora/v0.7-3b-l4/examples.jsonl"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)
assert "T4" not in gpu, "Change the runtime to L4 (Runtime -> Change runtime type)"
assert os.path.exists(EXAMPLES), f"{EXAMPLES} is missing"
shutil.rmtree(REPO, ignore_errors=True)
zipfile.ZipFile(f"{DRIVE}/legalgpt_colab.zip").extractall(REPO)
shutil.copy(EXAMPLES, "/content/examples.jsonl")   # read from Colab's own disk, not Drive
!pip install -q -r {REPO}/backend/scripts/colab/requirements-colab.txt

In [ ]:
def sh(cmd):
    """Run a command, stream its output, and stop the notebook if it fails."""
    env = {**os.environ, "TQDM_MININTERVAL": "60", "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode})")

out = f"{DRIVE}/lora/v0.7-1.5b-l4"
sh("python backend/scripts/train_lora.py --base-model Qwen/Qwen2.5-1.5B-Instruct --version v0.2 --no-judgments "
   "--examples-from /content/examples.jsonl --epochs 2 --batch-size 4 --grad-accum 2 --save-steps 20 "
   f"--out {out}")
print(sorted(os.listdir(out)))

In [ ]:
# stop the GPU so no more compute units are used (the adapter is in Drive)
from google.colab import runtime
runtime.unassign()